<a name="top"></a><img src="images/chisel_1024.png" alt="Chisel logo" style="width:480px;" />

# Module 5.1: The RV32I Instruction Set
**Prev: [A FIRRTL Transform Example](4.4_firrtl_add_ops_per_module.ipynb)**<br>
**Next: [The ALU, the Immediates and the Registers](5.2_alu_and_regfile.ipynb)**<br>

## Motivation
Module 5 builds a RISC-V processor in Chisel, one piece per notebook, in the sbt project under
`riscv/` (its design decisions are in [riscv/DESIGN.md](https://github.com/sysprog21/chisel-bootcamp/blob/dev/riscv/DESIGN.md)). Before writing hardware, this
notebook looks at what the processor must execute: the 40 instructions of RV32I, the base integer
instruction set of 32-bit RISC-V. You will see

- the 32 registers and their ABI names;
- the six instruction formats, and why the immediates are split up the way they are;
- a decoder that turns a 32-bit word into a Scala case class;
- a small assembler, which the next notebooks use to write test programs;

and then write a disassembler yourself.

Everything here is plain Scala from `riscv/src/main/scala/riscv/` (`Asm.scala` and `Isa.scala`): it
is the reference that the hardware decoder of the processor is checked against.

## Setup
The second cell compiles the `riscv/` project with sbt the first time it runs (about half a minute)
and adds its classes to this notebook.

In [ ]:
interp.load.module(os.pwd / "source" / "load-ivy.sc")

In [ ]:
interp.load.module(os.pwd / "source" / "load-riscv.sc")

In [ ]:
import chisel3._
import chisel3.util.BitPat
import riscv._

---
# Registers

RV32I has 32 registers of 32 bits, `x0` to `x31`, and the program counter `pc`. `x0` always reads
0, and writes to it are ignored: this gives a constant zero and a place to throw results away. The
other 31 are the same for the hardware; the *calling convention* (the ABI) gives each one a name
and a role, which assembly programs use:

| Register | ABI name | Role |
|---|---|---|
| `x0` | `zero` | always 0 |
| `x1` | `ra` | return address |
| `x2` | `sp` | stack pointer |
| `x3`, `x4` | `gp`, `tp` | global and thread pointer (unused here) |
| `x5`-`x7`, `x28`-`x31` | `t0`-`t6` | temporaries (a call may change them) |
| `x8`, `x9`, `x18`-`x27` | `s0`-`s11` | saved by the callee (`s0` is also `fp`) |
| `x10`-`x17` | `a0`-`a7` | arguments; `a0` and `a1` also return values (a call may change them) |

`Reg` turns either kind of name into a register number:

In [ ]:
println(Reg.names.zipWithIndex.map { case (n, i) => s"x$i=$n" }.mkString(" "))
println(s"a0 is x${Reg("a0")}, x10 is x${Reg("x10")}, fp is x${Reg("fp")}")

---
# The six instruction formats

Every RV32I instruction is one 32-bit word. The low 7 bits are the *opcode*, which selects the
format; the other fields are always in the same place when a format has them, so the hardware can
read the registers before it knows what the instruction is:

```
     31        25 24     20 19     15 14  12 11        7 6       0
R   |   funct7   |   rs2   |   rs1   |funct3|     rd    |  opcode |  register-register ALU ops
I   |      imm[11:0]       |   rs1   |funct3|     rd    |  opcode |  ALU ops with a constant, loads, jalr
S   | imm[11:5]  |   rs2   |   rs1   |funct3|  imm[4:0] |  opcode |  stores
B   |imm[12|10:5]|   rs2   |   rs1   |funct3|imm[4:1|11]|  opcode |  conditional branches
U   |               imm[31:12]              |     rd    |  opcode |  lui, auipc
J   |         imm[20|10:1|11|19:12]         |     rd    |  opcode |  jal
```

`rd` is the destination register, `rs1` and `rs2` the sources; `funct3` and `funct7` select the
operation within an opcode (for example `add` and `sub` differ only in `funct7`).
`Rv32i.encodings` lists the format, opcode, funct3 and funct7 of each instruction:

In [ ]:
for ((format, insts) <- Rv32i.encodings.groupBy(_._2.format).toSeq.sortBy(_._1.toString))
  println(s"$format: ${insts.keys.toSeq.sorted.mkString(" ")}")

---
# Immediates

The immediate is always sign-extended from its top bit, and that bit is always instruction bit 31,
so the hardware can sign-extend before it decodes the format. The rest of the immediate is placed
so that each immediate bit comes from as few instruction bits as possible across the formats; the
cost is the strange order of the B and J fields:

- **I**: `imm[11:0] = inst[31:20]`, from -2048 to 2047.
  The shifts by an immediate (`slli`, `srli`, `srai`) are the exception: `inst[24:20]` is an
  unsigned shift amount, 0 to 31, and `inst[31:25]` is a `funct7` (`0100000` for `srai`).
- **S**: the same 12 bits, but split around `rd`'s position (a store has no `rd`):
  `imm[11:5] = inst[31:25]`, `imm[4:0] = inst[11:7]`.
- **B**: a branch offset is a multiple of 2, so `imm[0]` is not stored and the 12 stored bits reach
  from -4096 to 4094. The sign bit (`inst[31]`, here `imm[12]`) and `imm[10:1]` are where the S
  format has them (`imm[10:5]` and `imm[4:1]`), and the place of the S format's `imm[0]`
  (`inst[7]`) holds `imm[11]`.
- **U**: `inst[31:12]` are the upper 20 bits, the low 12 are zero: `lui a0, 0x12345` puts
  `0x12345000` in `a0`, and an `addi` adds the low 12 bits. As `addi` sign-extends them, a value
  whose bit 11 is set needs the upper part rounded up: `0x12345800` is `lui 0x12346` plus
  `addi -2048` (the assembler's `li` and `%hi` do this).
- **J**: like B, `imm[0]` is not stored; the 20 stored bits reach ±1 MiB.

Here is the B-type immediate taken apart by hand, for `beq a0, a1, -8`:

In [ ]:
val word                                  = Rv32i.encode(Inst("beq", rs1 = Reg("a0"), rs2 = Reg("a1"), imm = -8))
def bits(x: Long, hi: Int, lo: Int): Long = (x >> lo) & ((1L << (hi - lo + 1)) - 1)
println(f"word: 0x$word%08x = ${word.toBinaryString.reverse.padTo(32, '0').reverse}")
val imm13 = bits(word, 31, 31) << 12 | bits(word, 7, 7) << 11 | bits(word, 30, 25) << 5 | bits(word, 11, 8) << 1
println(s"imm[12:0] = ${imm13.toBinaryString}, sign-extended: ${(imm13.toInt << 19) >> 19}")

---
# Decoding into a case class

`Rv32i.decode` turns a word into an `Inst`: the instruction's name, its register numbers and its
sign-extended immediate (fields that the format does not have are 0). It returns `None` for a word
that is not an RV32I instruction, and `Rv32i.encode` does the opposite:

In [ ]:
for (w <- Seq(0x00a00293L, 0x40208233L, 0xffc12503L, 0x00b42423L, 0x12345537L, 0x00000073L, 0x34011073L))
  println(f"0x$w%08x -> ${Rv32i.decode(w)}")
println(f"0x${Rv32i.encode(Inst("addi", rd = 5, imm = 10))}%08x")

The hardware decoder of the processor will use the bit patterns of `riscv.Instructions` instead
(the `BitPat`s of [Module 2.7](2.7_enums_and_bundles.ipynb)). The two descriptions are written
independently, and a test in `riscv/` checks that they agree: for each of the 40 instructions, 100
random instructions are encoded, matched against every `BitPat` (only their own one may match) and
decoded back to the same `Inst`. A similar check runs here the other way round: starting from
each `BitPat`, it fills the bits the pattern leaves free (`mask` has a 1 where the pattern fixes the
bit, `value` holds the fixed bits) with random ones, then decodes and re-encodes the word:

In [ ]:
val rnd = new scala.util.Random(0)
for ((name, pattern) <- Instructions.all; _ <- 1 to 100) {
  val word    = (rnd.nextLong() & ~pattern.mask.toLong | pattern.value.toLong) & 0xffffffffL
  val decoded = Rv32i.decode(word)
  assert(decoded.map(_.name).contains(name.toLowerCase), f"$name: 0x$word%08x decodes to $decoded")
  assert(Rv32i.encode(decoded.get) == word, f"$name: 0x$word%08x")
}
println("SUCCESS!!")

## What this course does with the 40 instructions

There are no CSRs (see [riscv/DESIGN.md](https://github.com/sysprog21/chisel-bootcamp/blob/dev/riscv/DESIGN.md)), so there are no traps either:

- `fence` orders memory accesses between harts and devices; with one in-order core it is a no-op.
- `ecall` and `ebreak` would trap to an operating system or a debugger. Here they **halt** the
  core, which reports why on its retire port; so do an illegal instruction, a misaligned load or
  store, and a branch or jump to an address that is not a multiple of 4.
  The test programs end with `ecall`, leaving their result in `a0`.
- The other 37 do what the specification says.

---
# The assembler

Writing words by hand is error-prone, and the course has no RISC-V compiler: like the
[Ripes](https://github.com/mortbopet/Ripes) simulator, it brings its own assembler, `riscv.Asm`,
written in Scala in `riscv/src/main/scala/riscv/Asm.scala` (read it: it is a two-pass assembler in
a few hundred lines). It reads the GNU syntax:

- one statement per line, `#` comments, labels (`loop:`) and numeric labels (`1:`, used as `1b`
  backwards or `1f` forwards);
- the sections `.text` and `.data` (and `.bss`), and data directives such as `.word`, `.byte`,
  `.string` and `.align`;
- the RV32I pseudo-instructions, which stand for one or two real instructions: `nop`, `li` (any
  32-bit value), `la` (the address of a label), `mv`, `not`, `neg`, `j`, `call`, `ret`, branches
  against zero (`beqz`, `bnez`, ...) and the reversed comparisons (`bgt`, `ble`, ...).

A branch or jump target is a label, and the assembler computes the offset (a plain number is
taken as the offset itself). `Asm(source)` returns the memory image, one word per element, for a
program that starts at `0x8000_0000`: the instructions of `.text`, then the `.data` words.

This is the Fibonacci program of `riscv/programs/src/fib.S`; its words are those of
`riscv/programs/hex/fib.hex`, which the processors of the next notebooks load:

In [ ]:
val fib = Asm("""
    li    t0, 10          # how many steps
    li    a0, 0           # fib(0)
    li    a1, 1           # fib(1)
loop:
    beqz  t0, done
    add   t1, a0, a1
    mv    a0, a1
    mv    a1, t1
    addi  t0, t0, -1
    j     loop
done:
    ecall
halt: j   halt
""")
val hex = os.read.lines(os.pwd / "riscv" / "programs" / "hex" / "fib.hex").map(java.lang.Long.parseLong(_, 16))
for ((w, i) <- fib.zipWithIndex) println(f"${0x80000000L + 4 * i}%08x: $w%08x  ${Rv32i.decode(w).get}")
assert(fib == hex)
println("SUCCESS!!")

A program with data: `la` loads the address of `table` (an `auipc` and an `addi`, so it works
wherever the program is), and `Asm.assemble` also returns the address of every label:

In [ ]:
val sum = Asm.assemble("""
    la    t0, table       # t0 = the address of table
    li    t1, 4           # four words
    li    a0, 0
1:  lw    t2, 0(t0)
    add   a0, a0, t2
    addi  t0, t0, 4
    addi  t1, t1, -1
    bnez  t1, 1b
    ecall
    .data
table: .word 1, 2, 3, 0x12345678
""")
println(sum.symbols)
for ((w, i) <- sum.words.zipWithIndex) {
  val address = sum.base + 4 * i
  val what    = if (address < sum.symbols("table")) Rv32i.decode(w).get.toString else "data"
  println(f"$address%08x: $w%08x  $what")
}

---
# Exercise: a disassembler

`Inst` prints as a case class; a disassembler prints it as assembly. Write
`disassemble(inst: Long): String`, which returns the assembly of `inst` in the syntax that `Asm`
reads, with ABI register names, one space after the name and `", "` between operands:

| Kind | Example |
|---|---|
| R | `add t1, a0, a1` |
| loads and `jalr` | `lw a0, -4(sp)` |
| other I (shifts too) | `addi t0, t0, -1` |
| S | `sw a1, 8(s0)` |
| B (the offset, in bytes) | `beq t0, zero, 24` |
| U (the 20 upper bits, in hex) | `lui a0, 0x12345` |
| J (the offset) | `jal zero, -20` |
| `ecall`, `ebreak`, `fence` | just the name |
| not RV32I | `unknown` |

Use `Rv32i.decode`, `Rv32i.encodings(name).format`, `Rv32i.isLoad` and `Reg.names`.

In [ ]:
def disassemble(inst: Long): String = {
  ??? // your code here
}

In [ ]:
val expected = Seq(
  0x00b50333L -> "add t1, a0, a1",
  0xffc12503L -> "lw a0, -4(sp)",
  0x000300e7L -> "jalr ra, 0(t1)",
  0xfff28293L -> "addi t0, t0, -1",
  0x41f55513L -> "srai a0, a0, 31",
  0x00b42423L -> "sw a1, 8(s0)",
  0x00028c63L -> "beq t0, zero, 24",
  0x12345537L -> "lui a0, 0x12345",
  0xfedff06fL -> "jal zero, -20",
  0x00000073L -> "ecall",
  0x0ff0000fL -> "fence",
  0x00000000L -> "unknown"
)
for ((word, text) <- expected) assert(disassemble(word) == text, f"0x$word%08x: got '${disassemble(word)}'")
// and the assembler reads the result back: every word of the program above survives the round trip
for (w <- fib) assert(Asm(disassemble(w), base = 0) == Seq(w), disassemble(w))
println("SUCCESS!!")

<div id="container"><section id="accordion"><div>
<input type="checkbox" id="check-1" />
<label for="check-1"><strong>Solution</strong></label>
<article>
<pre style="background-color:#f7f7f7">
def disassemble(inst: Long): String = Rv32i.decode(inst) match {
  case None =&gt; "unknown"
  case Some(i) =&gt;
    def r(n: Int) = Reg.names(n)
    Rv32i.encodings(i.name).format match {
      case _ if Set("ecall", "ebreak", "fence")(i.name) =&gt; i.name
      case Format.R                                       =&gt; s"${i.name} ${r(i.rd)}, ${r(i.rs1)}, ${r(i.rs2)}"
      case Format.I if Rv32i.isLoad(i.name) || i.name == "jalr" =&gt;
        s"${i.name} ${r(i.rd)}, ${i.imm}(${r(i.rs1)})"
      case Format.I =&gt; s"${i.name} ${r(i.rd)}, ${r(i.rs1)}, ${i.imm}"
      case Format.S =&gt; s"${i.name} ${r(i.rs2)}, ${i.imm}(${r(i.rs1)})"
      case Format.B =&gt; s"${i.name} ${r(i.rs1)}, ${r(i.rs2)}, ${i.imm}"
      case Format.U =&gt; f"${i.name} ${r(i.rd)}, 0x${i.imm &gt;&gt;&gt; 12}%x"
      case Format.J =&gt; s"${i.name} ${r(i.rd)}, ${i.imm}"
    }
}
</pre></article></div></section></div>

---
# You're done!

The [next notebook](5.2_alu_and_regfile.ipynb) builds the first pieces of hardware: the ALU, the
immediate generator and the register file.

[Return to the top.](#top)